[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/tabular-ml-practice/00_pandas_for_tabular/00_pandas_for_tabular.ipynb)

# 00. 이 시리즈에서 쓰는 pandas 문법

01~04번 노트북에는 `pd.to_datetime`, `get_dummies`, `dropna` 같은 pandas 문법이 계속 나옵니다.
**여기서는 그 문법 하나하나가 정확히 무슨 일을 하는지** 직접 실행해가며 확인합니다.

쓰는 데이터도 01~04번과 같은 `taxis`·`titanic`입니다. 그래서 여기서 본 코드가
뒤 노트북에 그대로 다시 나옵니다.

## 이 노트북을 쓰는 두 가지 방법

1. **처음부터 순서대로 읽기** — 20~30분이면 끝납니다. 01번으로 넘어갈 준비가 됩니다
2. **사전처럼 찾아보기** — 01~04번을 읽다가 모르는 문법이 나오면 아래 표에서 찾아 그 절만 보기

| 문법 | 하는 일 | 절 |
|---|---|---|
| `df["col"]`, `df[조건]` | 열 고르기, 행 거르기 | 2 |
| `.copy()` | 잘라낸 조각을 안전하게 수정하기 | 2 |
| `pd.to_datetime`, `.dt.hour` | 문자열을 시각으로 바꾸고 요일·시각 꺼내기 | 3 |
| `.isnull().sum()`, `.dropna()`, `.fillna()` | 빈 칸 세기 / 지우기 / 채우기 | 4 |
| `.value_counts()`, `.nunique()` | 값 종류와 개수 세기 | 5 |
| `.quantile()`, `.nlargest()` | 사분위수, 상위 N개 | 5 |
| `.groupby().mean()`, `.agg()`, `.transform()` | 그룹별 집계 | 6 |
| `pd.get_dummies()` | 문자열 범주를 0/1 컬럼으로 | 7 |
| `pd.cut()`, `.map()`, `.where()` | 구간 나누기, 값 바꾸기 | 8 |

## 선수 지식

`df.head()`, `df.info()`, `df.describe()`, 열 선택, `groupby` 기본은
[`ml-curriculum/00_python_essentials`](../../ml-curriculum/00_python_essentials/00_python_essentials.ipynb)의
**실습 2([Pandas](../../../glossary.md#pandas))** 에서 다룹니다. 이 노트북은 그다음 단계입니다.

**소요 시간**: 처음부터 읽으면 25~35분. 사전처럼 필요한 절만 찾아본다면 절마다 2~3분입니다.
무거운 학습이 없어 모든 셀이 몇 초 안에 끝납니다.

## 읽는 법

- 셀을 위에서부터 순서대로 실행하세요(`Shift + Enter`).
- **실행 결과는 저장되어 있지 않습니다.** 직접 실행해야 표가 나타납니다.
- 각 절 끝에 **"01~04번 어디서 쓰이는가"** 를 적어뒀습니다.
- 코드 셀 뒤의 **결과 읽는 법**은 그 셀의 출력을 어떻게 읽는지 알려줍니다.
  숫자가 예상과 다르면 거기부터 보세요.
- 낯선 용어는 [glossary.md](../../../glossary.md)에서 찾아보세요.
- **에러가 나거나 결과가 예상과 다르면** [troubleshooting.md](../../../troubleshooting.md)를 먼저 보세요.
  설치 실패, 한글 깨짐, `NameError`, API 키, GPU 설정처럼 여러 노트북에서 반복되는 문제를 모아뒀습니다.

## 막혔을 때 — 이 노트북에서 자주 나오는 증상

| 증상 | 원인 | 해볼 것 |
|---|---|---|
| 준비 셀에서 `URLError` / `HTTPError` | `sns.load_dataset()`은 데이터를 **인터넷에서 받아옵니다** | 네트워크 연결 확인. 사내망이라면 프록시 때문일 수 있습니다 |
| `KeyError: 'fare'` 같은 에러 | 컬럼 이름 오타, 또는 앞 절에서 그 컬럼을 이미 지움 | `df.columns`를 먼저 찍어보기 |
| `SettingWithCopyWarning`이 뜬다 | 잘라낸 조각에 `.copy()`를 안 붙이고 값을 바꿈 | **2절**에서 다루는 내용입니다. 그 절을 먼저 보세요 |
| `NameError: name 'trips' is not defined` | 준비 셀을 건너뛰고 중간 절부터 실행 | 맨 위 준비 셀을 먼저 실행 |

여기 없는 문제(설치 실패, 한글 깨짐, GPU 설정)는 [troubleshooting.md](../../../troubleshooting.md)에 모아뒀습니다.

## 준비 셀

라이브러리를 불러오고 실습용 데이터 두 개를 받아옵니다. **내용을 이해할 필요 없이 그냥 실행**하면 됩니다.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !pip install -q pandas seaborn matplotlib

두 번째 준비 셀입니다. 라이브러리를 불러오고 **실습용 데이터 두 개**를 받아옵니다.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)   # 컬럼이 많아도 생략하지 않고 보여줍니다

trips = sns.load_dataset("taxis")       # 뉴욕 택시 운행 기록
titanic = sns.load_dataset("titanic")   # 타이타닉 탑승자

print("trips  :", trips.shape)     # (행, 열)
print("titanic:", titanic.shape)

> `sns.load_dataset("taxis")`는 seaborn이 인터넷에서 예제 데이터를 받아 **DataFrame**으로 돌려줍니다.
> `.shape`는 `(행 수, 열 수)` 튜플입니다.

---

## 1. DataFrame과 Series — 표와 열 하나

pandas에는 자료형이 두 개뿐입니다. 이 둘을 구분하면 대부분의 혼란이 사라집니다.

| | 뜻 | 만들어지는 경우 |
|---|---|---|
| **DataFrame** | 표 전체 (2차원) | `sns.load_dataset(...)`, `df[["a", "b"]]` |
| **Series** | 열 하나 (1차원) | `df["a"]`, `df["a"].value_counts()` |

In [ ]:
print(type(trips))            # DataFrame — 표 전체
print(type(trips["fare"]))    # Series    — 열 하나

print()
print(trips[["distance", "fare"]].head(3))   # 대괄호 두 겹 -> DataFrame
print()
print(trips["fare"].head(3))                 # 대괄호 한 겹 -> Series

**결과 읽는 법**

**대괄호가 한 겹이면 Series, 두 겹이면 DataFrame**입니다. 겉보기엔 사소하지만,
scikit-learn에 넣을 때 `X`는 DataFrame(2차원), `y`는 Series(1차원)여야 하므로 중요합니다.

```python
X = df.drop(columns="duration")   # DataFrame
y = df["duration"]                # Series
```

02번 마지막의 `prepare_trips()` 함수가 정확히 이 형태로 돌려줍니다.

---

## 2. 행 거르기와 `.copy()`

### 2-1. 조건으로 행 고르기 (불리언 인덱싱)

`df[조건]`은 **조건이 `True`인 행만** 남깁니다. 조건 자체는 True/False로 이루어진 Series입니다.

In [ ]:
cond = trips["distance"] > 10          # True/False Series
print(cond.head(5))
print()
print("조건을 만족하는 행 수:", cond.sum())   # True를 1로 세면 개수가 됩니다
print()

long_trips = trips[cond]               # True인 행만 남김
print(long_trips.shape)

### 2-2. 조건 두 개 이상 — 괄호가 필수

`and` / `or` 대신 **`&`(그리고) / `|`(또는)** 를 쓰고, **각 조건을 반드시 괄호로 감쌉니다.**

```python
trips[(trips["distance"] > 10) & (trips["fare"] < 50)]     # ✅
trips[trips["distance"] > 10 & trips["fare"] < 50]         # ❌ 엉뚱하게 해석됨
```

`&`가 `>` 보다 먼저 계산되기 때문입니다. 괄호를 빼면 에러가 나거나, 더 나쁘게는
**에러 없이 틀린 결과**가 나옵니다.

In [ ]:
sel = trips[(trips["distance"] > 10) & (trips["fare"] < 50)]
print("거리 10마일 초과 & 요금 50달러 미만:", len(sel), "건")

# ~ 는 조건을 뒤집습니다 (NOT)
print("그 조건의 반대                    :", len(trips[~((trips["distance"] > 10) & (trips["fare"] < 50))]), "건")

### 2-3. `.copy()` — 왜 붙이는가

`trips[조건]`은 원본의 일부를 **잘라낸 조각**입니다. 이 조각을 수정하면 pandas가
`SettingWithCopyWarning`을 띄우고, 원본이 바뀔지 사본이 바뀔지 보장되지 않습니다.

**잘라낸 결과를 계속 수정할 거라면 `.copy()` 를 붙이세요.** 02번 전체가 이 방식을 씁니다.

```python
clean = trips[trips["distance"] > 0].copy()   # ✅ 독립된 사본
clean["new_col"] = 1                          # 안전
```

In [ ]:
clean = trips[trips["distance"] > 0].copy()
clean["is_long"] = clean["distance"] > 10     # 경고 없이 안전하게 추가됨

print(clean[["distance", "is_long"]].head(3))
print()
print("원본에는 새 컬럼이 없습니다:", "is_long" in trips.columns)

> **01~04번 어디서 쓰이는가**
> - 02번 2절: `trips[(trips["duration"] > 0) & (trips["speed"] < 60)].copy()` — 이상치 제거
> - 02번 2절: `titanic[~is_outlier].copy()` — IQR 이상치 제거

---

## 3. 새 열 만들기 — 파생 변수

### 3-1. 열끼리 계산하기

열 단위로 한 번에 계산됩니다. 반복문을 쓸 필요가 없습니다(**벡터화**).

In [ ]:
df = trips.copy()

df["fare_per_mile"] = df["fare"] / df["distance"]     # 열 ÷ 열
df["total_check"] = df["fare"] + df["tip"] + df["tolls"]

print(df[["distance", "fare", "fare_per_mile", "total_check", "total"]].head(3))

### 3-2. `pd.to_datetime` — 문자열을 시각으로

CSV나 원본 데이터에서 날짜·시각은 대개 **문자열**로 들어옵니다. 문자열끼리는 뺄셈이 안 되고,
"몇 시인지" 꺼낼 수도 없습니다. `pd.to_datetime()`이 이것을 **datetime 자료형**으로 바꿔줍니다.

```python
df["pickup"] = pd.to_datetime(df["pickup"])
```

바꾸고 나면 두 가지가 가능해집니다.

1. **뺄셈** → 두 시각의 차이(`Timedelta`)
2. **`.dt` 접근자** → 연·월·일·요일·시각을 꺼내기

In [ ]:
print("변환 전 자료형:", trips["pickup"].dtype)

df["pickup"] = pd.to_datetime(df["pickup"])
df["dropoff"] = pd.to_datetime(df["dropoff"])

print("변환 후 자료형:", df["pickup"].dtype)
print()
print(df["pickup"].head(3))

**결과 읽는 법**

**뺄셈 결과는 `Timedelta`** 라서 그대로는 숫자가 아닙니다.
`.dt.total_seconds()`로 **초 단위 숫자**로 바꾼 뒤 60으로 나누면 분이 됩니다.

In [ ]:
gap = df["dropoff"] - df["pickup"]
print("뺄셈 결과 자료형:", gap.dtype)
print(gap.head(3))
print()

df["duration"] = gap.dt.total_seconds() / 60      # 초 -> 분
print(df["duration"].head(3).round(2))

### 3-3. `.dt` 접근자 — 시각에서 정보 꺼내기

`.dt` 뒤에 원하는 것을 붙이면 됩니다. **모델은 "2019-03-23 20:21" 같은 값을 이해하지 못하지만,
"토요일 20시"는 이해할 수 있습니다.** 그래서 쪼개서 넣습니다.

| 코드 | 결과 |
|---|---|
| `.dt.year` / `.dt.month` / `.dt.day` | 연 / 월 / 일 |
| `.dt.hour` / `.dt.minute` | 시 / 분 |
| `.dt.dayofweek` | **요일 (0=월 … 6=일)** |
| `.dt.day_name()` | 요일 이름 (`Monday` …) |
| `.dt.date` | 날짜 부분만 |

In [ ]:
df["weekday"] = df["pickup"].dt.dayofweek     # 0=월 … 6=일
df["hour"] = df["pickup"].dt.hour             # 0~23

print(df[["pickup", "weekday", "hour"]].head(5))
print()
print("요일 이름으로 확인:", df["pickup"].dt.day_name().head(3).tolist())

> **01~04번 어디서 쓰이는가**
> - 네 노트북 모두 첫 셀: `duration`(예측 대상), `speed`, `weekday`, `hour`를 이렇게 만듭니다
> - 01번 1부: "원본 컬럼을 조합해 새 컬럼을 만드는 것을 **파생 변수**라고 한다"의 실제 코드

---

## 4. 결측치 — 빈 칸 다루기

pandas에서 빈 칸은 `NaN`(Not a Number)입니다. **scikit-learn 모델은 `NaN`이 들어오면 에러**를 내므로
반드시 처리해야 합니다.

### 4-1. 세기 — `isnull()`

`isnull()`은 각 칸이 비었는지를 True/False로 바꿉니다. 여기에 `.sum()`을 붙이면 컬럼별 개수,
`.mean()`을 붙이면 **비율**이 나옵니다(True=1이므로 평균이 곧 비율).

In [ ]:
print("컬럼별 결측치 개수")
print(titanic.isnull().sum())
print()
print("비율(%)")
print((titanic.isnull().mean() * 100).round(1))

컬럼이 많으면 전체를 훑기 번거롭습니다. **[결측치](../../../glossary.md#missing-value)가 있는 컬럼만** 골라 한 표로 만들어두면
01·02번에서 그대로 쓸 수 있습니다.

In [ ]:
# 결측치가 있는 컬럼만 골라 개수와 비율을 한 표로
missing = pd.DataFrame({
    "개수": titanic.isnull().sum(),
    "비율(%)": (titanic.isnull().mean() * 100).round(1),
})
missing[missing["개수"] > 0]

### 4-2. 지우기 — `dropna()`

| 코드 | 하는 일 |
|---|---|
| `df.dropna()` | **한 칸이라도** 비면 그 행 삭제 |
| `df.dropna(subset=["age"])` | `age`가 빈 행만 삭제 |
| `df.dropna(axis=1)` | 결측치가 있는 **컬럼**을 삭제 |

`df.dropna()`가 특히 위험합니다. 결측이 많은 컬럼 하나 때문에 멀쩡한 행이 대량으로 날아갑니다.

In [ ]:
print("원본                     :", len(titanic), "행")
print("dropna()                 :", len(titanic.dropna()), "행   ← deck 하나 때문에 80%가 사라짐")
print("age만 기준으로 dropna     :", len(titanic.dropna(subset=["age"])), "행")
print("deck 컬럼을 버린 뒤 dropna:", len(titanic.drop(columns="deck").dropna()), "행")

### 4-3. 채우기 — `fillna()`

| 코드 | 무엇으로 채우나 |
|---|---|
| `s.fillna(s.mean())` | 평균 |
| `s.fillna(s.median())` | **중앙값** (치우친 분포에 안전) |
| `s.fillna(s.mode()[0])` | 최빈값 (범주형) |
| `s.fillna("Unknown")` | 고정값 |

`mode()`는 최빈값이 여러 개일 수 있어 **Series를 돌려주므로 `[0]`으로 첫 값을 꺼내야** 합니다.

In [ ]:
age = titanic["age"]

print(f"원본        결측 {age.isnull().sum():3d}개  평균 {age.mean():.2f}  표준편차 {age.std():.2f}")
print(f"중앙값 대체 결측 {age.fillna(age.median()).isnull().sum():3d}개  "
      f"평균 {age.fillna(age.median()).mean():.2f}  표준편차 {age.fillna(age.median()).std():.2f}")
print()
print("embarked의 최빈값:", titanic["embarked"].mode()[0], "  (mode()는 Series를 돌려줍니다)")

> **01~04번 어디서 쓰이는가**
> - 01번 1부·2부: `isnull().sum()`으로 결측 현황 파악
> - 02번 1부: 비율이 낮아 `dropna()`로 삭제 / 02번 2부: `fillna()`로 대체하고 부작용 관찰

---

## 5. 값 세기와 요약

### 5-1. `value_counts()` — 각 값이 몇 번 나오는가

범주형 컬럼에서 가장 많이 쓰는 함수입니다.

| 옵션 | 효과 |
|---|---|
| `normalize=True` | 개수 대신 **비율** |
| `dropna=False` | 결측치도 하나의 값으로 세기 |
| `.head(10)` | 상위 10개만 |

In [ ]:
print(trips["pickup_borough"].value_counts(dropna=False))
print()
print("비율(%)")
print((trips["pickup_borough"].value_counts(normalize=True) * 100).round(1))

### 5-2. 고유값 — `nunique()` / `unique()`

- `nunique()` → 고유값이 **몇 개**인지 (숫자)
- `unique()` → 고유값 **목록** (배열)

범주형을 0/1 컬럼으로 펼칠 때 **고유값 개수만큼 컬럼이 늘어나므로** 미리 확인해야 합니다.

In [ ]:
for col in ["color", "payment", "pickup_borough", "pickup_zone"]:
    print(f"{col:16s} 고유값 {trips[col].nunique():3d}개  {trips[col].unique()[:4]}")

### 5-3. `quantile()` — 사분위수

`quantile(0.25)`는 "아래에서 25% 지점의 값"입니다. **[IQR](../../../glossary.md#iqr) [이상치](../../../glossary.md#outlier) 기준**을 만들 때 씁니다.

```
IQR = Q3 - Q1
이상치 = Q1 - 1.5×IQR 보다 작거나, Q3 + 1.5×IQR 보다 큰 값
```

In [ ]:
q1 = titanic["fare"].quantile(0.25)
q3 = titanic["fare"].quantile(0.75)
iqr = q3 - q1

print(f"Q1 {q1:.2f}  Q3 {q3:.2f}  IQR {iqr:.2f}")
print(f"이상치 기준: {q1 - 1.5 * iqr:.2f} 미만 또는 {q3 + 1.5 * iqr:.2f} 초과")
print()

# 한 번에 여러 분위수를 구할 수도 있습니다
print(titanic["fare"].quantile([0.25, 0.5, 0.75]))

### 5-4. `nlargest()` / `nsmallest()` — 상위·하위 N개 행

`sort_values(ascending=False).head(3)` 과 같지만 더 짧고 빠릅니다.
**이상치의 정체를 확인할 때** 유용합니다.

In [ ]:
tmp = trips.copy()
tmp["speed"] = tmp["distance"] / ((pd.to_datetime(tmp["dropoff"]) - pd.to_datetime(tmp["pickup"])).dt.total_seconds() / 3600)

print("평균 시속이 가장 높은 3건")
print(tmp.nlargest(3, "speed")[["distance", "speed"]].round(2))

### 5-5. `corr()` — 상관계수

수치형 컬럼끼리 얼마나 함께 움직이는지를 -1 ~ +1로 계산합니다.
**`numeric_only=True`를 넣지 않으면 문자열 컬럼 때문에 에러**가 납니다.

In [ ]:
corr = titanic.corr(numeric_only=True)

# survived 와의 상관만 뽑아 절댓값 순으로 정렬
print(corr["survived"].drop("survived").sort_values(key=abs, ascending=False).round(3))

> **01~04번 어디서 쓰이는가**
> - 01번 1부: `value_counts`, `nunique`, `nlargest`, `corr`로 데이터 파악
> - 02번 2부: `quantile`로 IQR 이상치 기준 계산

---

## 6. 그룹별로 묶어서 계산 — `groupby`

`ml-curriculum/00번`에서 `groupby(...).mean()`까지 봤습니다. 여기서는 실무에서 자주 쓰는
**두 가지 확장**을 봅니다.

### 6-1. 0/1 컬럼의 평균 = 비율

`survived`는 0과 1로만 이루어져 있으므로, **평균이 곧 생존율**입니다. 자주 쓰는 요령입니다.

In [ ]:
print("성별 생존율(%)")
print((titanic.groupby("sex")["survived"].mean() * 100).round(1))
print()
print("등급별 생존율(%)")
print((titanic.groupby("pclass")["survived"].mean() * 100).round(1))

### 6-2. `agg()` — 한 번에 여러 통계

`agg(["mean", "median", "count"])` 처럼 목록으로 넘기면 여러 지표를 한 표에 뽑습니다.

In [ ]:
trips.groupby("payment")["tip"].agg(["mean", "median", "count"]).round(3)

### 6-3. `transform()` — 그룹 통계를 원래 행 수만큼 돌려주기

`mean()`은 **그룹 수만큼**(2줄, 3줄) 결과를 주지만, `transform()`은 **원래 행 수만큼** 돌려줍니다.
그래서 **컬럼에 그대로 대입**할 수 있습니다. 그룹별 결측치 대체에 쓰입니다.

```python
# 등급·성별이 같은 사람들의 중앙값으로 age의 빈 칸을 채우기
titanic.groupby(["pclass", "sex"])["age"].transform(lambda s: s.fillna(s.median()))
```

In [ ]:
grouped_mean = titanic.groupby("pclass")["age"].mean()
transformed = titanic.groupby("pclass")["age"].transform("mean")

print("mean()      결과 길이:", len(grouped_mean), "(그룹 수)")
print("transform() 결과 길이:", len(transformed), "(원래 행 수)")
print()

filled = titanic.groupby(["pclass", "sex"])["age"].transform(lambda s: s.fillna(s.median()))
print("그룹별 중앙값으로 채운 뒤 결측:", filled.isnull().sum(), "개")

> **01~04번 어디서 쓰이는가**
> - 01번 2부: `groupby("sex")["survived"].mean()` — 성별 생존율
> - 02번 2부: `transform`으로 등급·성별 그룹의 중앙값 대체
> - 03번 해설: `agg`로 구간별 오차 집계

---

## 7. 범주형을 숫자로 — `pd.get_dummies`

모델은 `"Manhattan"` 같은 문자열을 계산할 수 없습니다. **범주마다 컬럼을 만들고 해당하면 1,
아니면 0**을 넣는 것을 [원-핫 인코딩](../../../glossary.md#one-hot-encoding)이라고 합니다.

| 원본 | → | `color_green` | `color_yellow` |
|---|---|---|---|
| yellow | | 0 | 1 |
| green | | 1 | 0 |

In [ ]:
sample = trips[["color", "payment", "distance"]].head(5)
print("변환 전")
print(sample)
print()
print("변환 후")
print(pd.get_dummies(sample, columns=["color", "payment"]))

### 7-1. `drop_first=True` — 컬럼 하나씩 빼기

`color_green`과 `color_yellow`는 **항상 합이 1**입니다. 하나만 알면 나머지가 결정되므로
한 컬럼은 잉여입니다. `drop_first=True`가 각 범주의 첫 컬럼을 하나씩 뺍니다.
**정보는 전혀 잃지 않습니다.**

In [ ]:
a = pd.get_dummies(sample, columns=["color", "payment"])
b = pd.get_dummies(sample, columns=["color", "payment"], drop_first=True)

print("drop_first=False :", a.shape[1], "개 컬럼", list(a.columns))
print("drop_first=True  :", b.shape[1], "개 컬럼", list(b.columns))

### 7-2. `select_dtypes` — 인코딩 대상 찾기

어떤 컬럼이 문자열인지 일일이 확인하는 대신, 자료형으로 골라낼 수 있습니다.

```python
df.select_dtypes(include="object").columns.tolist()   # 문자열 컬럼 목록
df.select_dtypes(include="number")                    # 숫자 컬럼만
```

인코딩이 끝난 뒤 **`object` 컬럼이 하나도 남지 않았는지 확인**하는 용도로도 씁니다.
하나라도 남아 있으면 모델에서 에러가 납니다.

In [ ]:
print("문자열 컬럼:", trips.select_dtypes(include="object").columns.tolist())
print()

encoded = pd.get_dummies(trips.select_dtypes(include="object"), drop_first=True)
print("인코딩 후 남은 object 컬럼:", encoded.select_dtypes(include="object").columns.tolist())
print("(빈 목록이면 정상입니다)")

> **01~04번 어디서 쓰이는가**
> - 02번 1부·2부: `get_dummies(..., drop_first=True)` — 전처리의 마지막 단계
> - 02번 5절: `select_dtypes`로 인코딩 대상 확인, 인코딩 후 검산

---

## 8. 값 바꾸기와 구간 나누기

### 8-1. `pd.cut()` — 연속값을 구간으로

숫자를 "~5분 / 5-10분 / 10-20분" 같은 **구간(범주)** 으로 묶습니다.
경계값 목록과 라벨을 함께 넘깁니다.

In [ ]:
tmp = trips.copy()
tmp["dur"] = (pd.to_datetime(tmp["dropoff"]) - pd.to_datetime(tmp["pickup"])).dt.total_seconds() / 60

tmp["구간"] = pd.cut(tmp["dur"], [0, 5, 10, 20, 40, 120],
                     labels=["~5분", "5-10분", "10-20분", "20-40분", "40분+"])

print(tmp["구간"].value_counts().sort_index())

### 8-2. `.map()` — 값을 다른 값으로 바꾸기

딕셔너리를 주면 **값 하나하나를 갈아끼웁니다.** True/False를 읽기 좋은 라벨로 바꿀 때 편합니다.

In [ ]:
flag = titanic["survived"].map({0: "사망", 1: "생존"})
print(flag.head(5).tolist())
print()
print(flag.value_counts())

### 8-3. `.where()` — 조건이 거짓인 곳만 바꾸기

`s.where(조건, 대체값)` 은 **조건이 참인 곳은 그대로 두고, 거짓인 곳만** 바꿉니다.
(`.mask()`는 정반대로 동작합니다.)

고유값이 너무 많은 범주형에서 **상위 N개만 남기고 나머지를 `"Other"`로 묶을 때** 씁니다.

In [ ]:
top10 = trips["pickup_zone"].value_counts().head(10).index
grouped = trips["pickup_zone"].where(trips["pickup_zone"].isin(top10), "Other")

print("원래 고유값:", trips["pickup_zone"].nunique(), "개")
print("묶은 뒤    :", grouped.nunique(), "개")
print()
print(grouped.value_counts().head(5))

> **01~04번 어디서 쓰이는가**
> - 02번 해설 문제 3: `where`로 상위 20개 지역 + `"Other"`
> - 03번 해설 문제 2: `pd.cut`으로 거리·시간 구간별 오차 분석
> - 01번 해설: `map`으로 그래프 라벨 만들기

---

## 9. 자주 보는 에러 세 가지

### `SettingWithCopyWarning`

```python
part = df[df["a"] > 0]
part["b"] = 1            # ⚠️ 경고
```
→ **`.copy()`를 붙이세요.** (2-3절)

### `ValueError: could not convert string to float`

문자열 컬럼을 그대로 모델에 넣었을 때 납니다.
→ **`get_dummies`로 인코딩**했는지, `select_dtypes(include="object")`가 비었는지 확인하세요. (7절)

### `ValueError: Input contains NaN`

결측치가 남아 있습니다.
→ **`isnull().sum()`으로 어디인지 확인하고 `dropna()` 또는 `fillna()`** 하세요. (4절)

---

## 확인 문제

정답은 바로 아래 셀에 있습니다. **먼저 직접 써본 뒤** 실행해보세요.

1. `trips`에서 `payment`가 `"cash"`이고 `distance`가 5마일 이상인 행이 몇 건인가요?
2. `titanic`의 `embarked` 결측치를 최빈값으로 채우세요.
3. `trips`의 승차 시각에서 **월(month)** 을 꺼내 새 컬럼으로 만드세요.
4. `titanic`을 `who`(man/woman/child) 기준으로 원-핫 인코딩하면 컬럼이 몇 개 늘어나나요?
5. `titanic`의 `age`를 `[0, 12, 20, 40, 60, 100]` 구간으로 나눠 구간별 생존율을 구하세요.

In [ ]:
# 1
print("1:", len(trips[(trips["payment"] == "cash") & (trips["distance"] >= 5)]), "건")

# 2
filled = titanic["embarked"].fillna(titanic["embarked"].mode()[0])
print("2: 채운 뒤 결측", filled.isnull().sum(), "개")

# 3
month = pd.to_datetime(trips["pickup"]).dt.month
print("3:", month.unique(), "(2019년 3월 데이터라 3만 나옵니다)")

# 4
print("4: who 고유값", titanic["who"].nunique(), "개 ->",
      pd.get_dummies(titanic[["who"]], columns=["who"]).shape[1], "개 컬럼")

# 5
band = pd.cut(titanic["age"], [0, 12, 20, 40, 60, 100],
              labels=["아동", "10대", "청년", "중년", "노년"])
print("5: 연령대별 생존율(%)")
print((titanic.groupby(band, observed=True)["survived"].mean() * 100).round(1))

---

## 정리

| 하고 싶은 일 | 코드 |
|---|---|
| 조건에 맞는 행만 | `df[(조건1) & (조건2)].copy()` |
| 문자열을 시각으로 | `pd.to_datetime(df["col"])` |
| 시각에서 요일·시각 꺼내기 | `.dt.dayofweek`, `.dt.hour` |
| 결측치 현황 | `df.isnull().sum()`, `df.isnull().mean()` |
| 결측치 삭제 / 대체 | `df.dropna(subset=[...])`, `s.fillna(s.median())` |
| 값 개수 / 비율 | `s.value_counts()`, `s.value_counts(normalize=True)` |
| 고유값 개수 | `s.nunique()` |
| 사분위수 | `s.quantile([0.25, 0.75])` |
| 그룹별 통계 | `df.groupby("col")["target"].mean()` / `.agg([...])` |
| 그룹 통계를 행 수만큼 | `.transform(...)` |
| 범주형 → 0/1 | `pd.get_dummies(df, columns=[...], drop_first=True)` |
| 문자열 컬럼 찾기 | `df.select_dtypes(include="object")` |
| 구간 나누기 | `pd.cut(s, [경계], labels=[...])` |
| 값 갈아끼우기 | `s.map({0: "a", 1: "b"})`, `s.where(조건, 대체값)` |

이 표에 있는 것만 알면 01~04번의 코드는 전부 읽힙니다.

---

다음 노트북: [01_eda_visualization.ipynb](../01_eda_visualization/01_eda_visualization.ipynb) —
이 문법으로 실제 데이터를 탐색합니다.

## 스스로 확인해보기

- [ ] `df["col"]`과 `df[["col"]]`의 결과가 어떻게 다른지 안다
- [ ] 조건 두 개를 `&`로 묶을 때 괄호가 필요한 이유를 안다
- [ ] 잘라낸 조각을 수정하기 전에 `.copy()`를 붙이는 이유를 안다
- [ ] `pd.to_datetime` 후 `.dt`로 무엇을 꺼낼 수 있는지 안다
- [ ] 결측치를 지울지 채울지 판단하는 기준을 하나 말할 수 있다
- [ ] `get_dummies(drop_first=True)`가 왜 필요한지 안다
- [ ] `groupby().mean()`과 `groupby().transform()`의 결과 길이가 왜 다른지 안다

막히는 항목이 있으면 그 절로 돌아가 셀을 다시 실행해보세요.
**이 노트북은 한 번에 다 외우는 자료가 아니라 01~04번을 보다가 찾아오는 사전입니다.**
